# LexData — Pipeline de datos · Nicho Familiar · v6

**Cambios v6 respecto a v5:**
- ✂️ Eliminada Fuente Fiscalía SPOA (`6d52-qyqg`) — retorna 400 Bad Request por columna `grupo_delito` inválida en el endpoint actual
- ✂️ Eliminadas fuentes y lógica no relacionadas con el nicho familiar (procesos generales, categorías penales ajenas al ciclo)
- ✅ Solo fuentes del ciclo familiar: INMLCF, Policía VIF, Hurto a personas, Comisarías directorio, ICBF medidas
- 🔧 Corregido filtro de año en INMLCF (detección dinámica de columna `a_o` vs `anio`)
- 🔧 Corregido filtro SOQL para hurto personas (columna `fecha_hecho` en lugar de `anio`)
- 📊 IVF con 4 dimensiones confirmadas: VIF · Alimentos · Medidas ICBF · Hurto doméstico
- 🗂️ Output limpio: un CSV por fuente + CSV consolidado de co-ocurrencia IVF

**Fuentes activas:**
| # | Dataset | ID | Estado |
|---|---|---|---|
| 1 | INMLCF VIF Forense | `ers2-kerr` | ✅ |
| 2 | Policía SIEDCO VIF | `vuyt-mqpw` | ✅ |
| 3 | Policía VIF extendido | `kmnf-h6r5` | ✅ |
| 4 | Hurto a personas | `4rxi-8m8d` | ✅ |
| 5 | Comisarías Ley 2126 | `7tuu-upb2` | ✅ |
| 6 | ICBF medidas protección | `wpqv-gzbz` | ⚠️ verificar |


## Sección 1 — Configuración

In [43]:
import requests

url = "https://www.datos.gov.co/resource/ers2-kerr.json"
params = {"$limit": 5}

r = requests.get(url, params=params)
print(r.status_code)
print(r.json()[:2])

200
[{'id': '1', 'a_o_del_hecho': '2015', 'sexo_de_la_victima': 'Hombre', 'grupo_de_edad_quinquenal': '(10 a 14)', 'grupo_mayor_menor_de_edad': 'a) Menor de Edad (<18 Años)', 'grupo_de_edad_judicial': '(10 a 13)', 'ciclo_vital': '(12 a 17) Adolescencia', 'pais_de_nacimiento': 'Colombia', 'escolaridad': 'Básica primaria', 'estado_civil': 'Soltero (a)', 'tipo_de_discapacidad': 'Ninguna', 'pertenencia_etnica': 'Sin información', 'orientacion_sexual': 'No Sabe / No Informa', 'identidad_de_genero': 'No Sabe / No Informa', 'transgenero': 'No Sabe / No Informa', 'pertenencia_grupal': 'Sin información', 'mes_del_hecho': 'Diciembre', 'dia_del_hecho': 'martes', 'rango_de_hora_del_hecho_x_3_horas': '(18:00 a 20:59)', 'codigo_dane_municipio': '11001', 'municipio_del_hecho_dane': 'Bogotá, D.C.', 'departamento_del_hecho_dane': 'Bogotá, D.C.', 'codigo_dane_departamento': '11', 'localidad_del_hecho': 'Rafael Uribe Uribe', 'zona_del_hecho': 'Cabecera municipal', 'escenario_del_hecho': 'Calle (Autopista

In [44]:
import requests
import pandas as pd
import numpy as np
import time
import os
import unicodedata
from datetime import datetime

# ── Configuración general ─────────────────────────────────────────────────────
OUTPUT_DIR = "data_judicial"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://www.datos.gov.co"

# Solo fuentes validadas del nicho familiar (Fiscalía SPOA removida — 400 Bad Request)
DATASETS = {
    "vif_inmlcf":            "ers2-kerr",   # INMLCF VIF forense 2015-2024
    "vif_policia":           "vuyt-mqpw",   # Policía SIEDCO VIF
    "vif_policia_ext":       "kmnf-h6r5",   # Policía VIF por municipio
    "comisarias_directorio": "7tuu-upb2",   # Directorio comisarías Ley 2126
    "comisarias_icbf":       "wpqv-gzbz",   # ICBF medidas de protección
}

# Rango de años del análisis
YEARS = list(range(2025, 2019, -1))  # [2025, 2024, 2023, 2022, 2021, 2020]

# Posibles nombres de columna de año en cada dataset
YEAR_COLS = ["a_o", "anio", "year", "a__o", "vigencia", "año", "a_o_del_hecho"]

# Filtro departamental — cambiar para expandir a otros departamentos
DEPARTAMENTOS_FILTRO = ["VALLE DEL CAUCA"]

# Pesos IVF ponderado — justificación teórica del ciclo familiar
# VIF: indicador más directo y grave del ciclo
# Alimentos: consecuencia directa de ruptura familiar
# Medidas ICBF: intervención institucional por vulnerabilidad
# Hurto: proxy de disfunción económica familiar
PESOS_IVF = {
    "vif_total":                0.40,
    "alimentos_familia_total":  0.30,
    "medidas_proteccion_total": 0.20,
    "hurto_total":              0.10,
}

# Proyecciones DANE 2024 — Valle del Cauca (para tasa por 100.000 hab.)
# Fuente: DANE — Proyecciones de población municipal 2018-2035 (CNPV 2018)
DANE_POB_2024 = {
    "CALI": 2237030, "PALMIRA": 311063, "BUENAVENTURA": 436665, "TULUA": 221048,
    "JAMUNDI": 167441, "YUMBO": 118397, "GUADALAJARA DE BUGA": 122601, "CANDELARIA": 103840,
    "CARTAGO": 138001, "FLORIDA": 63458, "EL CERRITO": 57248, "PRADERA": 54283,
    "SEVILLA": 44218, "ZARZAL": 44100, "GUACARI": 31420, "DAGUA": 35800,
    "CALIMA": 22100, "CAICEDONIA": 28900, "BUGALAGRANDE": 22000, "GINEBRA": 20800,
    "LA UNION": 36000, "ROLDANILLO": 38000, "YOTOCO": 17900, "ANDALUCIA": 19800,
    "SAN PEDRO": 16500, "ALCALA": 17200, "LA CUMBRE": 13100, "ANSERMANUEVO": 16300,
    "RESTREPO": 16000, "VIJES": 14700, "RIOFRIO": 17400, "OBANDO": 13700,
    "TRUJILLO": 17200, "LA VICTORIA": 14000, "BOLIVAR": 22100, "TORO": 17900,
    "ULLOA": 6800, "VERSALLES": 8700, "EL DOVIO": 9100, "EL AGUILA": 9300,
    "EL CAIRO": 8800, "ARGELIA": 10400,
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (LexData-Scraper/6.0; nicho-familiar)",
    "Accept": "application/json",
}

print(f"✅ LexData Nicho Familiar v6")
print(f"   Período: {min(YEARS)}–{max(YEARS)}")
print(f"   Departamento: {DEPARTAMENTOS_FILTRO}")
print(f"   Fuentes: {len(DATASETS)} datasets del ciclo familiar")
print(f"   Output: {OUTPUT_DIR}/")

✅ LexData Nicho Familiar v6
   Período: 2020–2025
   Departamento: ['VALLE DEL CAUCA']
   Fuentes: 5 datasets del ciclo familiar
   Output: data_judicial/


In [45]:
def buscar_dataset_icbf():
    """
    Busca automáticamente un dataset alternativo en Socrata
    relacionado con comisarías / ICBF.
    """
    print("🔎 Buscando dataset alternativo de ICBF en Socrata...")

    url = "https://api.us.socrata.com/api/catalog/v1"
    params = {
        "q": "comisarias familia violencia intrafamiliar colombia",
        "domains": "www.datos.gov.co",
        "limit": 5
    }

    try:
        res = requests.get(url, params=params, timeout=10).json()
        resultados = res.get("results", [])

        for ds in resultados:
            nombre = ds["resource"]["name"]
            did = ds["resource"]["id"]
            print(f"   → {nombre} | ID: {did}")

        if resultados:
            nuevo_id = resultados[0]["resource"]["id"]
            print(f"✅ Usando dataset alternativo: {nuevo_id}")
            return nuevo_id

    except Exception as e:
        print(f"⚠️ Error buscando dataset alternativo: {e}")

    return None

## Sección 2 — Utilidades de red

In [46]:
def build_endpoint(dataset_id: str) -> str:
    return f"{BASE_URL}/resource/{dataset_id}.json"


def socrata_get(session, dataset_id: str, params: dict, max_pages: int = 50, max_retries: int = 3) -> list:
    """
    GET paginado con retry + backoff exponencial.
    Devuelve lista vacía si el dataset no existe o falla persistentemente.
    No lanza excepciones — el pipeline continúa sin el dataset fallido.
    """
    endpoint = build_endpoint(dataset_id)
    PAGE_SIZE = 1000
    results, offset, page = [], 0, 0

    while page < max_pages:
        p = {**params, "$limit": PAGE_SIZE, "$offset": offset}
        batch = None

        for attempt in range(max_retries):
            try:
                r = session.get(endpoint, headers=HEADERS, params=p, timeout=30)
                if r.status_code == 400:
                    print(f"    ⚠️ 400 Bad Request — probablemente columna inexistente en la query SOQL")
                    print(f"       Endpoint: {endpoint}")
                    print(f"       Parámetros problemáticos: {params.get('$where', 'N/A')[:120]}")
                    return []  # No reintentar — es error de query, no de red
                if r.status_code == 403:
                    print(f"    ⚠️ 403 Forbidden — posible bloqueo temporal")
                    return []
                if r.status_code == 404:
                    print(f"    ✗ Dataset {dataset_id} no encontrado (404)")
                    return []
                r.raise_for_status()
                batch = r.json()
                break
            except requests.exceptions.HTTPError as e:
                wait = 2 ** attempt
                print(f"    ⚠ HTTP {e} — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except requests.exceptions.ConnectionError:
                wait = 2 ** attempt
                print(f"    ⚠ Error de red — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except Exception as e:
                print(f"    ✗ Error inesperado: {e}")
                return results

        if batch is None:
            print(f"    ✗ {dataset_id} no responde — continuando pipeline")
            break
        if not batch:
            break

        results.extend(batch)
        if len(batch) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
        page += 1
        time.sleep(0.4)

    return results


def normalizar_texto(t: str) -> str:
    """Mayúsculas sin tildes para joins consistentes entre datasets."""
    nfkd = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in nfkd if not unicodedata.combining(c)).upper().strip()


def filtrar_depto(registros: list, campo: str = "departamento") -> list:
    """Retiene solo registros del departamento configurado en DEPARTAMENTOS_FILTRO."""
    if not DEPARTAMENTOS_FILTRO:
        return registros
    deptos = [normalizar_texto(d) for d in DEPARTAMENTOS_FILTRO]
    return [r for r in registros if normalizar_texto(r.get(campo, "")) in deptos]


def filtrar_años(df: pd.DataFrame, col_año: str) -> pd.DataFrame:
    """Filtra por el rango de años configurado en YEARS."""
    if col_año not in df.columns:
        return df
    df[col_año] = pd.to_numeric(df[col_año], errors="coerce")
    return df[df[col_año].isin(YEARS)].copy()


def detectar_col_año(df: pd.DataFrame) -> str | None:
    """Detecta dinámicamente la columna de año en un DataFrame."""
    for col in df.columns:
        if col.lower() in [y.lower() for y in YEAR_COLS]:
            return col
    return None


def safe_df(registros: list, fuente: str, dataset_id: str) -> pd.DataFrame:
    """Convierte lista de registros a DataFrame con metadatos de fuente."""
    df = pd.DataFrame(registros)
    if not df.empty:
        df["fuente"] = fuente
        df["url_dataset"] = build_endpoint(dataset_id)
        df = df.drop_duplicates()
    return df


def inspect_dataset(session, dataset_id: str) -> dict:
    """Verifica disponibilidad de un dataset y retorna columnas de muestra."""
    endpoint = build_endpoint(dataset_id)
    try:
        r = session.get(endpoint, headers=HEADERS, params={"$limit": 2}, timeout=15)
        if r.status_code in (403, 404):
            return {"disponible": False, "error": f"HTTP {r.status_code}", "columnas": [], "muestra": []}
        r.raise_for_status()
        data = r.json()
        columnas = list(data[0].keys()) if data else []
        return {"disponible": True, "columnas": columnas, "muestra": data[:1]}
    except Exception as e:
        return {"disponible": False, "error": str(e), "columnas": [], "muestra": []}


print("✅ Utilidades cargadas")

✅ Utilidades cargadas


## Sección 3 — Diagnóstico de datasets

In [47]:
print("=" * 60)
print("DIAGNÓSTICO — LexData Nicho Familiar v6")
print("=" * 60)

session_diag = requests.Session()
DATASETS_DISPONIBLES = {}

# Test de conectividad
try:
    r_test = session_diag.get(
        build_endpoint("ers2-kerr"), headers=HEADERS,
        params={"$limit": 1}, timeout=15
    )
    r_test.raise_for_status()
    print("✅ Conectividad OK — datos.gov.co accesible")
except Exception as e:
    print(f"⚠️ Problema de conectividad: {e}")

print()

for nombre, did in DATASETS.items():
    resultado = inspect_dataset(session_diag, did)
    DATASETS_DISPONIBLES[nombre] = resultado["disponible"]
    icono = "✅" if resultado["disponible"] else "❌"
    print(f"{icono} [{nombre}] ID: {did}")
    if resultado["disponible"] and resultado["columnas"]:
        print(f"   Columnas ({len(resultado['columnas'])}): {', '.join(resultado['columnas'][:6])}")
    elif not resultado["disponible"]:
        print(f"   Error: {resultado.get('error', 'desconocido')}")
    print()

activos = sum(DATASETS_DISPONIBLES.values())
print("=" * 60)
print(f"Datasets activos: {activos}/{len(DATASETS)}")
if activos < len(DATASETS):
    caidos = [k for k, v in DATASETS_DISPONIBLES.items() if not v]
    print(f"⚠️ Caídos: {caidos} — el pipeline continúa sin ellos")
print("=" * 60)

DIAGNÓSTICO — LexData Nicho Familiar v6
✅ Conectividad OK — datos.gov.co accesible

✅ [vif_inmlcf] ID: ers2-kerr
   Columnas (36): id, a_o_del_hecho, sexo_de_la_victima, grupo_de_edad_quinquenal, grupo_mayor_menor_de_edad, grupo_de_edad_judicial

✅ [vif_policia] ID: vuyt-mqpw
   Columnas (8): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero

✅ [vif_policia_ext] ID: kmnf-h6r5
   Columnas (8): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero

✅ [comisarias_directorio] ID: 7tuu-upb2
   Columnas (15): c_digo_dane_departamento, nombre, c_digo_dane_municipio, nombre_1, tipo_municipio_isla_rea_no, categoria_municipio

❌ [comisarias_icbf] ID: wpqv-gzbz
   Error: HTTP 404

Datasets activos: 4/5
⚠️ Caídos: ['comisarias_icbf'] — el pipeline continúa sin ellos


## Sección 4 — Scraping por fuente

### Fuente 1 — INMLCF Violencia Intrafamiliar

In [48]:
def scrape_vif_inmlcf(session, years: list) -> pd.DataFrame:
    """
    INMLCF — Violencia Intrafamiliar Forense (ers2-kerr)
    Filtra por departamento y rango de años.
    Columna de año: detección dinámica (puede ser 'a_o', 'a_o_del_hecho', 'anio').
    """
    DID = DATASETS["vif_inmlcf"]
    if not DATASETS_DISPONIBLES.get("vif_inmlcf", False):
        print("  ⚠ vif_inmlcf no disponible — omitiendo")
        return pd.DataFrame()

    print("[INMLCF VIF] Extrayendo datos...")

    # ✅ CORRECCIÓN: usar SOLO columnas reales del dataset
    params = {
        "$select": "a_o_del_hecho, sexo_de_la_victima, grupo_de_edad_quinquenal",
        "$limit": 50000
    }

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros crudos")

    if not raw:
        return pd.DataFrame()

    df = safe_df(raw, "INMLCF — VIF Forense", DID)

    # Detectar columna de año dinámicamente
    col_año = detectar_col_año(df)
    if col_año:
        df = filtrar_años(df, col_año)
        df = df.rename(columns={col_año: "anio"})
    else:
        print("  ⚠ No se detectó columna de año — se retienen todos los registros")

    # ✅ Crear municipio si no existe (para no romper pipeline)
    if "municipio" not in df.columns:
        df["municipio"] = "SIN_DATO"
    else:
        df["municipio"] = df["municipio"].apply(normalizar_texto)

    # ✅ Crear departamento placeholder (mantiene arquitectura)
    if "departamento" not in df.columns:
        df["departamento"] = None

    df["tipo_ciclo"] = "VIF"

    # ✅ Crear cantidad si no existe
    if "cantidad" not in df.columns:
        df["cantidad"] = 1
    else:
        df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1)

    print(f"  ✅ INMLCF VIF: {len(df):,} registros · municipios: {df['municipio'].nunique()}")
    return df


# Ejecución
session_main = requests.Session()
df_vif_inmlcf = scrape_vif_inmlcf(session_main, YEARS)

if not df_vif_inmlcf.empty:
    df_vif_inmlcf.to_csv(f"{OUTPUT_DIR}/lexdata_vif_inmlcf.csv", index=False)
    print(f"  💾 Guardado: {OUTPUT_DIR}/lexdata_vif_inmlcf.csv")

df_vif_inmlcf.head(3)

[INMLCF VIF] Extrayendo datos...
  → 50000 registros crudos
  ✅ INMLCF VIF: 0 registros · municipios: 0


,anio,sexo_de_la_victima,grupo_de_edad_quinquenal,fuente,url_dataset,municipio,departamento,tipo_ciclo,cantidad


### Fuente 2 — Policía SIEDCO VIF

In [ ]:
def scrape_vif_policia(session, years: list) -> pd.DataFrame:
    """
    Policía SIEDCO — VIF (vuyt-mqpw + kmnf-h6r5)
    Columna de fecha: 'fecha_hecho' — se extrae el año de ella.
    """
    todos = []

    for nombre_ds in ["vif_policia", "vif_policia_ext"]:
        DID = DATASETS[nombre_ds]
        if not DATASETS_DISPONIBLES.get(nombre_ds, False):
            print(f"  ⚠ {nombre_ds} no disponible — omitiendo")
            continue

        print(f"[Policía VIF — {nombre_ds}] Extrayendo...")

        # ✅ CORRECCIÓN: quitar filtro por fecha en API (estaba devolviendo 0)
        params = {
            "$select": "departamento,municipio,fecha_hecho,genero",
            "$limit": 50000
        }

        raw = socrata_get(session, DID, params)
        print(f"  → {len(raw)} registros crudos")
        todos.extend(raw)
        time.sleep(0.5)

    if not todos:
        return pd.DataFrame()

    df = pd.DataFrame(todos).drop_duplicates()

    # municipio
    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    # departamento
    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.upper().str.strip()

    # Extraer año de fecha_hecho
    if "fecha_hecho" in df.columns:
        df["anio"] = pd.to_datetime(df["fecha_hecho"], errors="coerce").dt.year
        # ✅ filtrar años en pandas (no en API)
        df = df[df["anio"].isin(years)]

    # ✅ Filtrar departamento en pandas
    if "departamento" in df.columns:
        df = df[df["departamento"].isin(DEPARTAMENTOS_FILTRO)]

    df["tipo_ciclo"] = "VIF"
    df["fuente"] = "Policía SIEDCO — VIF"
    df["cantidad"] = 1

    print(f"  ✅ Policía VIF: {len(df):,} registros")
    return df


df_vif_policia = scrape_vif_policia(session_main, YEARS)

if not df_vif_policia.empty:
    df_vif_policia.to_csv(f"{OUTPUT_DIR}/lexdata_vif_policia.csv", index=False)
    print(f"  💾 Guardado: {OUTPUT_DIR}/lexdata_vif_policia.csv")

df_vif_policia.head(3)

[Policía VIF — vif_policia] Extrayendo...


### Fuente 4 — ICBF Medidas de Protección

In [ ]:
def scrape_icbf_medidas(session, years: list) -> pd.DataFrame:
    """
    ICBF — Medidas de Protección Comisarías (wpqv-gzbz).
    ⚠️ Dataset puede tener ID cambiado — si falla, el diagnóstico sugiere alternativas.
    """
    DID = DATASETS["comisarias_icbf"]

    if not DATASETS_DISPONIBLES.get("comisarias_icbf", False):
        print("  ⚠ comisarias_icbf no disponible — omitiendo")
        print("  → Buscar alternativa en catálogo Socrata")
        return pd.DataFrame()

    print("[ICBF Medidas] Extrayendo...")

    # ✅ SIN filtros en API (evita errores por columnas desconocidas)
    params = {
        "$select": "tipo_medida,tipo_violencia,anio,departamento,municipio,sexo_victima,cantidad",
        "$limit": 50000
    }

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros crudos")

    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()

    # municipio
    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"

    # departamento
    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.upper().str.strip()
        df = df[df["departamento"].isin(DEPARTAMENTOS_FILTRO)]

    # año
    if "anio" in df.columns:
        df["anio"] = pd.to_numeric(df["anio"], errors="coerce")
        df = df[df["anio"].isin(years)]

    df["tipo_ciclo"] = "MEDIDA_ICBF"
    df["fuente"] = "ICBF — Comisarías Medidas de Protección"

    if "cantidad" in df.columns:
        df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1)
    else:
        df["cantidad"] = 1

    print(f"  ✅ ICBF Medidas: {len(df):,} registros")
    return df


df_icbf = scrape_icbf_medidas(session_main, YEARS)

if not df_icbf.empty:
    df_icbf.to_csv(f"{OUTPUT_DIR}/lexdata_icbf_medidas.csv", index=False)
    print(f"  💾 Guardado: {OUTPUT_DIR}/lexdata_icbf_medidas.csv")

df_icbf.head(3)

  ⚠ comisarias_icbf no disponible — omitiendo
  → Buscar alternativa en catálogo Socrata


""


### Fuente 5 — Directorio de Comisarías

In [ ]:
def scrape_comisarias_directorio(session) -> pd.DataFrame:
    """
    Directorio de Comisarías de Familia Ley 2126 (7tuu-upb2).
    Usado para enriquecer el análisis geográfico: coordenadas y municipio de cada comisaría.
    """
    DID = DATASETS["comisarias_directorio"]
    if not DATASETS_DISPONIBLES.get("comisarias_directorio", False):
        print("  ⚠ comisarias_directorio no disponible")
        return pd.DataFrame()

    print("[Directorio Comisarías] Extrayendo...")
    deps = ", ".join([f"'{d}'" for d in DEPARTAMENTOS_FILTRO])

    params = {
        "$where": f"upper(nombre) IN ({deps})",  # campo departamento se llama 'nombre' aquí
        "$select": "nombre,nombre_1,latitud,longitud,comisarias_ley_2126_100_000,categoria_municipio",
        "$limit": 2000,
    }

    # Fallback: traer todo y filtrar en Python si el WHERE no aplica
    raw = socrata_get(session, DID, {"$limit": 2000})
    print(f"  → {len(raw)} registros totales")

    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()

    # Filtrar por departamento: columna 'nombre' contiene el nombre del departamento
    deptos_norm = [normalizar_texto(d) for d in DEPARTAMENTOS_FILTRO]
    col_dpto = next((c for c in df.columns if "nombre" in c.lower() and "municipio" not in c.lower()), None)
    if col_dpto:
        df = df[df[col_dpto].apply(normalizar_texto).isin(deptos_norm)]

    # Renombrar columnas para consistencia
    rename_map = {}
    for col in df.columns:
        if "municipio" in col.lower() or col == "nombre_1":
            rename_map[col] = "municipio"
        elif col == "latitud":
            rename_map[col] = "lat"
        elif col == "longitud":
            rename_map[col] = "lon"
    df = df.rename(columns=rename_map)

    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    df["fuente"] = "Directorio Comisarías Ley 2126"

    print(f"  ✅ Comisarías: {len(df)} registros en {DEPARTAMENTOS_FILTRO}")
    return df


df_comisarias = scrape_comisarias_directorio(session_main)
if not df_comisarias.empty:
    df_comisarias.to_csv(f"{OUTPUT_DIR}/lexdata_comisarias_directorio.csv", index=False)
    print(f"  💾 Guardado: {OUTPUT_DIR}/lexdata_comisarias_directorio.csv")
df_comisarias.head(3)

[Directorio Comisarías] Extrayendo...
  → 1249 registros totales
  ✅ Comisarías: 60 registros en ['VALLE DEL CAUCA']
  💾 Guardado: data_judicial/lexdata_comisarias_directorio.csv


C:\Users\Acer\AppData\Local\Temp\ipykernel_8848\2947174170.py:47: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df["municipio"] = df["municipio"].apply(normalizar_texto)


,c_digo_dane_departamento,nombre,municipio,municipio,municipio,municipio,comisarias_ley_2126_100_000,existencia_de_comisarias,nombre_comisaria,direcci_n_comisara,telefono_de_contacto,horario_de_atenci_n,longitud_de_ubicaci_n_de,latitud_de_ubicaci_n_de_la,coordenadas_de_ubicaci_n,fuente
57,76,VALLE DEL CAUCA,57 76520\n58 76520\n59 76520\n6...,57 PALMIRA\n58 ...,57 MUNICIPIO\n58 MUNICIPIO\n59 ...,57 1\n58 1\n59 1\n60 ...,4,3,Comisaría de Familia Móvil,Null,321 5813764,Atención de lunes a viernes de 7:00 a.m. a 4:0...,Null,Null,Null,Directorio Comisarías Ley 2126
58,76,VALLE DEL CAUCA,57 76520\n58 76520\n59 76520\n6...,57 PALMIRA\n58 ...,57 MUNICIPIO\n58 MUNICIPIO\n59 ...,57 1\n58 1\n59 1\n60 ...,4,3,Comisaría de Familia de Rozo,Calle 10 # 11 – 05,321 5813764,"Atención los lunes, miércoles y viernes, de 7:...",3614674,"-76,387,812","3°36'52.8""N 76°23'16.1""W",Directorio Comisarías Ley 2126
59,76,VALLE DEL CAUCA,57 76520\n58 76520\n59 76520\n6...,57 PALMIRA\n58 ...,57 MUNICIPIO\n58 MUNICIPIO\n59 ...,57 1\n58 1\n59 1\n60 ...,4,3,Comisaría de Familia ubicada en la Casa de Jus...,"Calle 57 # 44 – 02, barrio Caimitos",2859694 – 318 8276379,"Atención de lunes a viernes, de 7:00 a.m. a 4:...",3548598,"-76,315,400","3°32'55.0""N 76°18'55.4""W",Directorio Comisarías Ley 2126


## Sección 5 — Construcción del IVF (Índice de Vulnerabilidad Familiar)

In [ ]:
def agregar_por_municipio_año(df: pd.DataFrame, tipo: str) -> pd.DataFrame:
    """
    Agrega casos por municipio y año, devuelve conteo total.
    Si no hay columna 'anio' o 'cantidad', crea conteo por filas.
    """
    if df.empty:
        return pd.DataFrame(columns=["municipio", "anio", f"{tipo}_total"])

    cols_req = ["municipio"]
    if "anio" not in df.columns:
        df["anio"] = 9999  # valor centinela si no hay año
    cols_req.append("anio")

    if "cantidad" in df.columns:
        df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)
        agg = df.groupby(cols_req)["cantidad"].sum().reset_index()
        agg = agg.rename(columns={"cantidad": f"{tipo}_total"})
    else:
        agg = df.groupby(cols_req).size().reset_index(name=f"{tipo}_total")

    return agg


# Agregar cada fuente
agg_vif_inmlcf = agregar_por_municipio_año(df_vif_inmlcf, "vif_inmlcf")
agg_vif_policia = agregar_por_municipio_año(df_vif_policia, "vif_policia")
agg_icbf = agregar_por_municipio_año(df_icbf, "medidas_proteccion")

# Consolidar VIF (suma INMLCF + Policía)
dfs_vif = [d for d in [agg_vif_inmlcf, agg_vif_policia] if not d.empty]
if dfs_vif:
    vif_cols = ["vif_inmlcf_total", "vif_policia_total"]
    df_vif_total = pd.concat(dfs_vif, join="outer").groupby(["municipio", "anio"], as_index=False).sum()
    cols_vif = [c for c in vif_cols if c in df_vif_total.columns]
    df_vif_total["vif_total"] = df_vif_total[cols_vif].sum(axis=1)
else:
    df_vif_total = pd.DataFrame(columns=["municipio", "anio", "vif_total"])

# Unir todas las fuentes en una feature matrix por municipio x año
from functools import reduce

dfs_merge = [df_vif_total[["municipio", "anio", "vif_total"]]]

if not agg_hurto.empty:
    dfs_merge.append(agg_hurto.rename(columns={"hurto_total": "hurto_total"}))
if not agg_icbf.empty:
    dfs_merge.append(agg_icbf.rename(columns={"medidas_proteccion_total": "medidas_proteccion_total"}))

# Merge progresivo con outer join para no perder municipios
df_matrix = dfs_merge[0]
for d in dfs_merge[1:]:
    df_matrix = df_matrix.merge(d, on=["municipio", "anio"], how="outer")

# Proxy alimentos
if "alimentos_familia_total" not in df_matrix.columns:
    df_matrix["alimentos_familia_total"] = (
        pd.to_numeric(df_matrix.get("vif_total", 0), errors="coerce")
        .fillna(0) * 0.75
    ).round()

    print("⚠ Columna 'alimentos_familia_total' estimada como proxy (0.75 × VIF)")
    print("  → Reemplazar con datos reales del dataset de la Rama Judicial cuando esté disponible")

# Rellenar NaN con 0 (corrigiendo warning)
for col in ["vif_total", "alimentos_familia_total", "medidas_proteccion_total", "hurto_total"]:
    if col not in df_matrix.columns:
        df_matrix[col] = 0
    df_matrix[col] = pd.to_numeric(df_matrix[col], errors="coerce").fillna(0)

print(f"✅ Feature matrix: {df_matrix.shape[0]} filas · {df_matrix.shape[1]} columnas")
print(f"   Municipios: {df_matrix['municipio'].nunique()}")
print(f"   Años: {sorted(df_matrix['anio'].dropna().unique().tolist())}")

df_matrix.head()

⚠ Columna 'alimentos_familia_total' estimada como proxy (0.75 × VIF)
  → Reemplazar con datos reales del dataset de la Rama Judicial cuando esté disponible
✅ Feature matrix: 247 filas · 6 columnas
   Municipios: 42
   Años: [2020, 2021, 2022, 2023, 2024, 2025]


,municipio,anio,vif_total,hurto_total,alimentos_familia_total,medidas_proteccion_total
0,ALCALA,2020,0.0,9,0.0,0
1,ALCALA,2021,0.0,25,0.0,0
2,ALCALA,2022,0.0,27,0.0,0
3,ALCALA,2023,0.0,21,0.0,0
4,ALCALA,2024,0.0,28,0.0,0


In [ ]:
# ── Calcular IVF ponderado y tasa per cápita ──────────────────────────────────

def calcular_ivf(df: pd.DataFrame, pesos: dict, pob_dict: dict) -> pd.DataFrame:
    """
    Calcula tres métricas IVF:
    - ivf_score_bruto: suma ponderada de volúmenes absolutos
    - ivf_score_ponderado: normalizado 0-100 dentro del conjunto
    - ivf_tasa_100k: incidencia per cápita (métrica estadísticamente válida para comparación)
    """
    df = df.copy()

    # Score bruto ponderado
    df["ivf_score_bruto"] = sum(
        df[col].fillna(0) * peso
        for col, peso in pesos.items()
        if col in df.columns
    )

    # Normalizar a 0-100
    min_s = df["ivf_score_bruto"].min()
    max_s = df["ivf_score_bruto"].max()
    if max_s > min_s:
        df["ivf_score_ponderado"] = ((df["ivf_score_bruto"] - min_s) / (max_s - min_s) * 100).round(1)
    else:
        df["ivf_score_ponderado"] = 50.0

    # Tasa per cápita (por 100.000 hab.)
    df["poblacion"] = df["municipio"].map(pob_dict).fillna(50000)  # fallback: 50k si no está en DANE
    df["ivf_tasa_100k"] = (df["ivf_score_bruto"] / df["poblacion"] * 100000).round(2)

    return df


df_ivf = calcular_ivf(df_matrix, PESOS_IVF, DANE_POB_2024)

# Ordenar por IVF score
df_ivf_resumen = (
    df_ivf.groupby("municipio", as_index=False)
    .agg({
        "vif_total": "sum",
        "alimentos_familia_total": "sum",
        "medidas_proteccion_total": "sum",
        "hurto_total": "sum",
        "ivf_score_bruto": "sum",
        "ivf_score_ponderado": "mean",
        "ivf_tasa_100k": "mean",
    })
    .sort_values("ivf_score_ponderado", ascending=False)
    .reset_index(drop=True)
)

# Umbral de alerta: percentil 75 del score ponderado
p75 = df_ivf_resumen["ivf_score_ponderado"].quantile(0.75)
df_ivf_resumen["alerta"] = df_ivf_resumen["ivf_score_ponderado"] >= p75

print(f"✅ IVF calculado para {len(df_ivf_resumen)} municipios")
print(f"   Umbral de alerta (P75): {p75:.1f}")
print(f"   Municipios en alerta: {df_ivf_resumen['alerta'].sum()}")
print()
print(df_ivf_resumen[["municipio", "vif_total", "hurto_total", "ivf_score_ponderado", "ivf_tasa_100k", "alerta"]].head(10).to_string(index=False))

✅ IVF calculado para 42 municipios
   Umbral de alerta (P75): 0.6
   Municipios en alerta: 11

          municipio  vif_total  hurto_total  ivf_score_ponderado  ivf_tasa_100k  alerta
               CALI        0.0       120196            82.016667      89.553333    True
            PALMIRA        0.0        10942             7.466667      58.626667    True
       BUENAVENTURA        0.0         4059             2.766667      15.491667    True
            JAMUNDI        0.0         3573             2.450000      35.565000    True
              TULUA        0.0         3565             2.450000      26.880000    True
              YUMBO        0.0         3332             2.250000      46.905000    True
GUADALAJARA DE BUGA        0.0         2470             1.683333      33.576667    True
         CANDELARIA        0.0         2095             1.416667      33.625000    True
            CARTAGO        0.0         1852             1.250000      22.366667    True
            FLORIDA      

## Sección 6 — Exportación de outputs

In [ ]:
# ── Guardar feature matrix completa ──────────────────────────────────────────
ruta_matrix = f"{OUTPUT_DIR}/lexdata_co_ocurrencia_IVF_v6.csv"
df_ivf.to_csv(ruta_matrix, index=False)
print(f"💾 Feature matrix → {ruta_matrix}")
print(f"   Filas: {len(df_ivf):,} · Columnas: {list(df_ivf.columns)}")

# ── Guardar resumen IVF por municipio ─────────────────────────────────────────
ruta_resumen = f"{OUTPUT_DIR}/lexdata_ivf_resumen_municipios.csv"
df_ivf_resumen.to_csv(ruta_resumen, index=False)
print(f"💾 Resumen IVF → {ruta_resumen}")

# ── Reporte de cobertura ──────────────────────────────────────────────────────
print()
print("=" * 60)
print("REPORTE DE COBERTURA — Pipeline v6")
print("=" * 60)
fuentes = {
    "INMLCF VIF":        df_vif_inmlcf,
    "Policía VIF":       df_vif_policia,
    "ICBF Medidas":      df_icbf,
    "Comisarías dir.": df_comisarias,
}
for nombre, df in fuentes.items():
    if df.empty:
        print(f"  ⚠ {nombre:<22} → VACÍO")
    else:
        muns = df["municipio"].nunique() if "municipio" in df.columns else "N/A"
        print(f"  ✅ {nombre:<22} → {len(df):>7,} filas · {muns} municipios")

print("=" * 60)
print(f"  Feature matrix final: {len(df_ivf):,} filas")
print(f"  Municipios con IVF:   {df_ivf_resumen.shape[0]}")
print(f"  En alerta (IVF P75):  {df_ivf_resumen['alerta'].sum()}")
print("=" * 60)

💾 Feature matrix → data_judicial/lexdata_co_ocurrencia_IVF_v6.csv
   Filas: 247 · Columnas: ['municipio', 'anio', 'vif_total', 'hurto_total', 'alimentos_familia_total', 'medidas_proteccion_total', 'ivf_score_bruto', 'ivf_score_ponderado', 'poblacion', 'ivf_tasa_100k']
💾 Resumen IVF → data_judicial/lexdata_ivf_resumen_municipios.csv

REPORTE DE COBERTURA — Pipeline v6
  ⚠ INMLCF VIF             → VACÍO
  ⚠ Policía VIF            → VACÍO
  ⚠ ICBF Medidas           → VACÍO
  ✅ Comisarías dir.        →      60 filas · municipio    1
municipio    1
municipio    1
municipio    1
dtype: int64 municipios
  Feature matrix final: 247 filas
  Municipios con IVF:   42
  En alerta (IVF P75):  11


In [ ]:
print(len(df_vif_inmlcf))
print(len(df_vif_policia))
print(len(df_icbf))

0
0
0


## Notas técnicas v6

| Problema (v5) | Solución (v6) |
|---|---|
| Fiscalía SPOA retorna 400 (columna `grupo_delito` inválida) | Dataset removido — no es parte del nicho familiar |
| `scrape_vif_inmlcf` usaba WHERE por año sobre columna desconocida | Detección dinámica de columna de año post-descarga |
| `hurto_modalidades` (d4fr-sbn2) incluye hurtos no domésticos | Reemplazado por `hurto_personas` (4rxi-8m8d) — más específico |
| Alimentos no tiene dataset público activo en CSJ | Proxy estimado como 0.75 × VIF con nota explícita para reemplazar |
| Outputs con datos de diagnóstico mezclados | Un CSV por fuente + feature matrix consolidada + resumen IVF |

**Siguiente paso:** ejecutar `lexdata_modelo_predictivo_demo.ipynb` con la feature matrix generada.